In [ ]:

This notebook inspects the Django database configuration, `.env` contents, and connection parameters to find non-UTF-8 bytes affecting `python manage.py migrate`.
</VSCode.Cell>
<VSCode.Cell language="python">from pathlib import Path
from pprint import pprint

BASE_DIR = Path(r'C:/Users/dmuhe/Documents/DJANGO/denunciations_app')
settings_path = BASE_DIR / 'denunciations_app' / 'settings.py'
print('Settings path:', settings_path)
with settings_path.open('r', encoding='utf-8', errors='replace') as f:
    settings_text = f.read()
print('Settings snippet:')
print(settings_text.splitlines()[:80])

# Extract DATABASES section
start = settings_text.find("DATABASES = {")
end = settings_text.find("# Password validation")
print('\nDATABASES block:')
print(settings_text[start:end].strip())
</VSCode.Cell>
<VSCode.Cell language="python">env_path = BASE_DIR / '.env'
print('Env path:', env_path)

# Read raw bytes to check for non-UTF-8 content
raw = env_path.read_bytes()
print('raw bytes length:', len(raw))
print('first 160 bytes:', raw[:160])
print('\nByte values above 0x7f:')
print([ (i, hex(b)) for i,b in enumerate(raw) if b >= 0x80 ])

# Decode safely with replacement and show text
text = raw.decode('utf-8', errors='replace')
print('\nDecoded .env text:')
print(text)
</VSCode.Cell>
<VSCode.Cell language="python">from decouple import config
import os

# Load environment variables from .env if necessary
os.environ['DJANGO_SETTINGS_MODULE'] = 'denunciations_app.settings'

try:
    from decouple import AutoConfig
    config_loader = AutoConfig(search_path=BASE_DIR)
    db_params = {
        'ENGINE': config_loader('DB_ENGINE', default='django.db.backends.postgresql'),
        'NAME': config_loader('DB_NAME', default='denunciations_app'),
        'USER': config_loader('DB_USER', default='postgres'),
        'PASSWORD': config_loader('DB_PASSWORD', default=''),
        'HOST': config_loader('DB_HOST', default='127.0.0.1'),
        'PORT': config_loader('DB_PORT', default='5432'),
    }
    print('Loaded DB params:')
    pprint(db_params)
except Exception as exc:
    print('Error loading config:', exc)
</VSCode.Cell>
<VSCode.Cell language="python">import psycopg2

params = {
    'dbname': 'Min_Emploi_RDC',
    'user': 'postgres',
    'password': os.environ.get('DB_PASSWORD', ''),
    'host': '127.0.0.1',
    'port': 5432,
}

print('Testing parameter safety:')
for k,v in params.items():
    if isinstance(v, str):
        print(k, repr(v), [hex(ord(c)) for c in v])

try:
    conn = psycopg2.connect(**params)
    conn.close()
    print('Connection succeeded')
except Exception as exc:
    print('Connection test failed:', exc)
